# Customer Churn: Responsible Baseline Notebook

**Purpose:** Educational demonstration for the Rab Tech Academy task. This is not a deployment-ready model.

> **Critical limitation:** The supplied dataset contains only 12 rows. Metrics from a small holdout will be highly unstable and must not be treated as evidence of real-world performance. Confirm dataset provenance, label definition, permissions, and feature timing before any operational use.

## Plan
1. Inspect the dataset and quality checks.
2. Establish a majority-class baseline.
3. Demonstrate a simple ML pipeline with stratified cross-validation only if feasible.
4. Report limitations and responsible-use safeguards.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("customer-churn-training.csv")  # Keep CSV beside this notebook.
df = pd.read_csv(DATA_PATH)
df.head()


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [2]:
print("Shape:", df.shape)
print("\\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\\nMissing values:")
display(df.isna().sum().to_frame("missing"))
print("Exact duplicate rows:", df.duplicated().sum())
print("Unique customer IDs:", df["customer_id"].nunique(), "of", len(df))
print("\\nTarget balance:")
display(df["churned"].value_counts(dropna=False).rename_axis("churned").to_frame("count"))


Shape: (12, 7)
\nData types:


,dtype
customer_id,object
tenure_months,int64
support_tickets,int64
monthly_spend_inr,int64
last_login_days,int64
plan_type,object
churned,int64


\nMissing values:


,missing
customer_id,0
tenure_months,0
support_tickets,0
monthly_spend_inr,0
last_login_days,0
plan_type,0
churned,0


Exact duplicate rows: 0
Unique customer IDs: 12 of 12
\nTarget balance:


,count
churned,
0,7
1,5


## Quality review

The provided file has 12 rows, no missing values, no exact duplicate rows, and 12 unique customer IDs. The target has 7 zeros and 5 ones. These checks do not establish that records are representative or labels correct.

`customer_id` is an identifier and must not be a model feature. Confirm the exact meaning of `churned`, the time window for each feature, and that features were recorded before the prediction date.


In [3]:
target = "churned"
id_col = "customer_id"
feature_cols = [c for c in df.columns if c not in [target, id_col]]
X = df[feature_cols].copy()
y = df[target].astype(int)

print("Features used:", feature_cols)
print("Class counts:", y.value_counts().to_dict())


Features used: ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type']
Class counts: {0: 7, 1: 5}


## Non-ML baseline: majority class

The majority-class baseline predicts the most common label in the training data. It is a transparent reference point, not a useful customer intervention policy by itself. On a larger dataset, compare it with a domain-approved business rule as well.


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

# A tiny stratified split is shown only for teaching; metrics are unstable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

print("Test rows:", len(y_test))
print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, base_pred, labels=[0, 1]))
print(classification_report(y_test, base_pred, labels=[0, 1], zero_division=0))
print("Balanced accuracy:", balanced_accuracy_score(y_test, base_pred))


Test rows: 4
Confusion matrix [ [TN, FP], [FN, TP] ]:
[[2 0]
 [2 0]]
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         2
           1       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4

Balanced accuracy: 0.5


## ML demonstration: preprocessing + logistic regression

This pipeline handles categorical `plan_type` and numeric fields. Cross-validation is used only as a demonstration; with 12 records, folds and scores are extremely sensitive to individual rows. Do not select a production model from these results.


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000))
])

# With only 5 positive and 7 negative rows, use 3 folds for a demonstration.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0)
}
cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, error_score="raise")
pd.DataFrame({k: v for k, v in cv_results.items() if k.startswith("test_")}).agg(["mean", "std"]).T


,mean,std
test_balanced_accuracy,1.0,0.0
test_precision,1.0,0.0
test_recall,1.0,0.0
test_f1,1.0,0.0


## Responsible interpretation and next steps

- Do not deploy this model based on 12 rows.
- Verify data provenance, permission/license, collection method, and the churn-label definition.
- Define the prediction date and future outcome window. Confirm every feature is available before prediction; investigate leakage.
- Acquire a larger, representative, permissioned, time-stamped dataset.
- Compare against a documented non-ML rule and a majority-class baseline.
- Choose thresholds by considering the costs of false positives and false negatives.
- Evaluate calibration and group-level fairness only when data volume and lawful, appropriate attributes support it.
- If ever piloted, require human review and optional, non-coercive outreach. Never use the score to deny service or apply punitive pricing.
- Maintain a standard-support fallback and suspend the model if data quality, performance, privacy permission, or disparity thresholds fail.
